In [2]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

from models import *
use_gpu = torch.cuda.is_available()
batch_size = 128
model_name = "vgg16_quant_struct"
model = VGG16_quant()
use_gpu = torch.cuda.is_available()
device = torch.device("cuda" if use_gpu else "cpu") 
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

In [3]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))


testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [9]:
print_freq = 390 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input = input.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input = input.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


In [4]:
import os
import glob
import shutil
import torch

def save_checkpoint(state, is_best, fdir, best_prec):
    os.makedirs(fdir, exist_ok=True)
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)

    if is_best:
        # Remove previous best checkpoint
        for old_file in glob.glob(os.path.join(fdir, 'model_best_prec=*.pth.tar')):
            os.remove(old_file)

        # Save new best checkpoint with accuracy in filename
        best_path = os.path.join(fdir, f'model_best_prec={best_prec:.3f}.pth.tar')
        shutil.copyfile(filepath, best_path)

In [105]:
model = VGG16_quant().cuda()
fdir = f"final_model/final_model_checkpoints/"
checkpoint = torch.load(
    os.path.join(fdir, 'model_best.pth.tar'),
    map_location='cuda',
    weights_only=False
)
device = torch.device("cuda") 
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(),momentum=0.9)
model.load_state_dict(checkpoint['state_dict'])
optimizer.load_state_dict(checkpoint['optimizer'])

model.eval()
model.cuda()

prec = validate(testloader, model, criterion)
print(f"Model Test Accuracy Before Pruning: {prec}%")

Test: [0/79]	Time 0.129 (0.129)	Loss 0.3051 (0.3051)	Prec 94.531% (94.531%)
 * Prec 92.540% 
Model Test Accuracy Before Pruning: 92.54%


In [12]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune

def prune_channels(model, n_input=0, n_output=0, layer_idx=27):
    """
    Prune input/output channels of a quantized Conv2d in place.
    Counts specify the TOTAL channels to prune, not additional channels.
    """

    conv = model.features[layer_idx]

    if not isinstance(conv, nn.Conv2d) or not hasattr(conv, "weight_quant"):
        raise TypeError("Selected layer must be a quantized Conv2d.")

    if not (0 <= n_input < conv.in_channels and
            0 <= n_output < conv.out_channels):
        raise ValueError("Invalid pruning counts.")

    def get_active():
        mask = (conv.weight_mask if hasattr(conv, "weight_mask")
                else torch.ones_like(conv.weight))

        active_in = mask.ne(0).any(dim=(0, 2, 3))
        active_out = mask.ne(0).any(dim=(1, 2, 3))

        return active_in, active_out

    active_in, active_out = get_active()

    already_in = conv.in_channels - int(active_in.sum())
    already_out = conv.out_channels - int(active_out.sum())

    if n_input < already_in or n_output < already_out:
        raise ValueError(
            "Cannot restore pruned channels. Reload original checkpoint."
        )

    # Prune input channels
    if n_input > already_in:
        prune.ln_structured(
            conv, name="weight",
            amount=n_input - already_in,
            dim=1, n=1
        )

    # Prune output channels
    if n_output > already_out:
        prune.ln_structured(
            conv, name="weight",
            amount=n_output - already_out,
            dim=0, n=1
        )

    # Ensure quantization preserves zeroed weights
    if (n_input or n_output) and not hasattr(conv, "_quant_prune_hook"):

        def mask_quantized(_module, _inputs, weight_q):
            return weight_q * conv.weight_mask

        conv._quant_prune_hook = conv.weight_quant.register_forward_hook(
            mask_quantized
        )

    active_in, active_out = get_active()

    info = {
        "pruned_input": (~active_in).nonzero(as_tuple=True)[0].tolist(),
        "pruned_output": (~active_out).nonzero(as_tuple=True)[0].tolist(),
        "active_input": active_in.nonzero(as_tuple=True)[0].tolist(),
        "active_output": active_out.nonzero(as_tuple=True)[0].tolist()
    }

    print(f"Pruned input channels: {info['pruned_input']}")
    print(f"Pruned output channels: {info['pruned_output']}")
    print(f"Active array: {len(info['active_input'])} x "
          f"{len(info['active_output'])}")

    return info

In [106]:
# 2 input channels, 0 output channels
#info = prune_channels(model, n_input=2, n_output=0)

# # OR: 1 input channel, 1 output channel
# info = prune_channels(model, n_input=1, n_output=1)

# Prune 7 input channels and 7 output channels, leaves only one 3x3 kernel, (one PE in the systolic array)
info = prune_channels(model, n_input=7, n_output=7)

Pruned input channels: [0, 1, 3, 4, 5, 6, 7]
Pruned output channels: [0, 1, 2, 3, 4, 5, 7]
Active array: 1 x 1


In [107]:
print(info)

{'pruned_input': [0, 1, 3, 4, 5, 6, 7], 'pruned_output': [0, 1, 2, 3, 4, 5, 7], 'active_input': [2], 'active_output': [6]}


In [108]:
prec = validate(testloader, model, criterion)
print(prec)

Test: [0/79]	Time 0.057 (0.057)	Loss 4.2727 (4.2727)	Prec 16.406% (16.406%)
 * Prec 13.860% 
13.86


In [109]:
# Evaluate the pruned model on the SAME test set as the baseline above.
# The quantizer hook stays active on every forward pass.
model.eval()
correct_pruned = 0
conv = model.features[27]
prec = validate(testloader, model, criterion)
print(f"Test Accuracy: {prec}%")
print(f"After pruning ({len(info['pruned_input'])} input channels) and ({len(info['pruned_output'])} output channels)")
print(f"Before fine-tuning")
# Verify the effective quantized weights remain exactly zero after inference.
with torch.no_grad():
    # Fresh forward pass
    inputs, _ = next(iter(testloader))
    model(inputs[:1].to(device))

    # Get CURRENT quantized weights
    w_q = conv.weight_q.detach()
    mask = conv.weight_mask.bool()

    # Check all masked weights
    nonzero = torch.count_nonzero(w_q[~mask]).item()
    print("Nonzero pruned weights:", nonzero)

    # Check individual channels
    for c in info["pruned_input"]:
        count = torch.count_nonzero(w_q[:, c, :, :]).item()
        print(f"Input channel {c}: {count}")

    for c in info["pruned_output"]:
        count = torch.count_nonzero(w_q[c, :, :, :]).item()
        print(f"Output channel {c}: {count}")

    assert nonzero == 0, "Pruned quantized weights are NOT zero!"

print("Verified: all pruned quantized weights are zero.")

Test: [0/79]	Time 0.043 (0.043)	Loss 4.2727 (4.2727)	Prec 16.406% (16.406%)
 * Prec 13.860% 
Test Accuracy: 13.86%
After pruning (7 input channels) and (7 output channels)
Before fine-tuning
Nonzero pruned weights: 0
Input channel 0: 0
Input channel 1: 0
Input channel 3: 0
Input channel 4: 0
Input channel 5: 0
Input channel 6: 0
Input channel 7: 0
Output channel 0: 0
Output channel 1: 0
Output channel 2: 0
Output channel 3: 0
Output channel 4: 0
Output channel 5: 0
Output channel 7: 0
Verified: all pruned quantized weights are zero.


In [110]:
def adjust_learning_rate(optimizer, epoch):
    adjust_list = [10, 20, 30]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1     

In [111]:
# Create a fresh optimizer after pruning
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=3e-4
)

max_epochs = 40
best_prec = 0
for epoch in range(0, max_epochs):
    adjust_learning_rate(optimizer, epoch)
    train(trainloader, model, criterion, optimizer, epoch)
        
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)
    

Epoch: [0][0/391]	Time 0.289 (0.289)	Data 0.060 (0.060)	Loss 2.6392 (2.6392)	Prec 23.438% (23.438%)
Epoch: [0][390/391]	Time 0.022 (0.024)	Data 0.000 (0.000)	Loss 0.3710 (0.4332)	Prec 90.000% (85.016%)
Validation starts
Test: [0/79]	Time 0.042 (0.042)	Loss 0.3639 (0.3639)	Prec 87.500% (87.500%)
 * Prec 86.930% 
Epoch: [1][0/391]	Time 0.076 (0.076)	Data 0.054 (0.054)	Loss 0.2165 (0.2165)	Prec 93.750% (93.750%)
Epoch: [1][390/391]	Time 0.022 (0.023)	Data 0.000 (0.000)	Loss 0.1404 (0.2194)	Prec 95.000% (92.810%)
Validation starts
Test: [0/79]	Time 0.051 (0.051)	Loss 0.3895 (0.3895)	Prec 88.281% (88.281%)
 * Prec 87.420% 
Epoch: [2][0/391]	Time 0.083 (0.083)	Data 0.059 (0.059)	Loss 0.2184 (0.2184)	Prec 90.625% (90.625%)
Epoch: [2][390/391]	Time 0.021 (0.023)	Data 0.000 (0.000)	Loss 0.1757 (0.1800)	Prec 93.750% (94.042%)
Validation starts
Test: [0/79]	Time 0.045 (0.045)	Loss 0.2883 (0.2883)	Prec 92.188% (92.188%)
 * Prec 87.990% 
Epoch: [3][0/391]	Time 0.088 (0.088)	Data 0.062 (0.062)	Loss 

In [114]:
fdir = f"final_model_struct/final_model_struct_checkpoints_1x1"
best_prec = validate(testloader, model, criterion)
is_best = True
save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
        }, is_best, fdir, best_prec)

Test: [0/79]	Time 0.163 (0.163)	Loss 0.2479 (0.2479)	Prec 93.750% (93.750%)
 * Prec 91.070% 


In [120]:
#Test After fine-tuning
model.eval()
correct_pruned = 0
conv = model.features[27]
prec = validate(testloader, model, criterion)
print(f"Test Accuracy: {prec}%")
print(f"After pruning ({len(info['pruned_input'])} input channels) and ({len(info['pruned_output'])} output channels)")
print(f"AFTER fine-tuning")
# Verify the effective quantized weights remain exactly zero after inference.
with torch.no_grad():
    # Fresh forward pass
    inputs, _ = next(iter(testloader))
    model(inputs[:1].to(device))

    # Get CURRENT quantized weights
    w_q = conv.weight_q.detach()
    mask = conv.weight_mask.bool()

    # Check all masked weights
    nonzero = torch.count_nonzero(w_q[~mask]).item()
    print("Nonzero pruned weights:", nonzero)

    # Check individual channels
    for c in info["pruned_input"]:
        count = torch.count_nonzero(w_q[:, c, :, :]).item()
        print(f"Input channel {c}: {count}")

    for c in info["pruned_output"]:
        count = torch.count_nonzero(w_q[c, :, :, :]).item()
        print(f"Output channel {c}: {count}")

    assert nonzero == 0, "Pruned quantized weights are NOT zero!"

print("Verified: all pruned quantized weights are zero.")

Test: [0/79]	Time 0.286 (0.286)	Loss 0.2479 (0.2479)	Prec 93.750% (93.750%)
 * Prec 91.070% 
Test Accuracy: 91.07%
After pruning (7 input channels) and (7 output channels)
AFTER fine-tuning
Nonzero pruned weights: 0
Input channel 0: 0
Input channel 1: 0
Input channel 3: 0
Input channel 4: 0
Input channel 5: 0
Input channel 6: 0
Input channel 7: 0
Output channel 0: 0
Output channel 1: 0
Output channel 2: 0
Output channel 3: 0
Output channel 4: 0
Output channel 5: 0
Output channel 7: 0
Verified: all pruned quantized weights are zero.


In [116]:
conv = model.features[27]

with torch.no_grad():
    weight_q = conv.weight_quant(conv.weight)

    compact_weights = weight_q[
        info["active_output"]
    ][:, info["active_input"], :, :].contiguous()

print(compact_weights.shape)

torch.Size([1, 1, 3, 3])


In [117]:
print("Active weight positions:",
      conv.weight_mask.sum().item())

print("Total nonzero quantized weights:",
      torch.count_nonzero(conv.weight_q).item())

assert conv.weight_mask.sum().item() == 9
assert torch.count_nonzero(conv.weight_q).item() <= 9

Active weight positions: 9.0
Total nonzero quantized weights: 9


In [118]:
handle = conv.register_forward_hook(
    lambda module, inputs, output: torch.zeros_like(output)
)

try:
    acc = validate(testloader, model, criterion)
    print("Accuracy with squeezed layer disabled:", acc)
finally:
    handle.remove()

Test: [0/79]	Time 0.221 (0.221)	Loss 7.6606 (7.6606)	Prec 7.812% (7.812%)
 * Prec 10.000% 
Accuracy with squeezed layer disabled: 10.0


In [143]:
model.eval()
w_bit = 4
layer = model.features[27]
weight_q = layer.weight_q 
w_alpha = layer.weight_quant.wgt_alpha.data.item()  
w_delta = w_alpha/(2**(w_bit-1)-1)   
weight_int = weight_q / w_delta 
print(weight_int[6][2]) #Only output channel 6's kernel corresponding to input channel 2 is active in the entire layer (rest are zeroed)
print("Total nonzero quantized integer weights:",
      torch.count_nonzero(weight_int).item())

tensor([[ 7.,  7.,  7.],
        [ 7.,  7., -7.],
        [ 7.,  7., -7.]], device='cuda:0', grad_fn=<SelectBackward0>)
Total nonzero quantized integer weights: 9


In [144]:

input_batch, _ = next(iter(testloader))
input_batch = input_batch.cuda()

with torch.no_grad():
    model(input_batch)

In [145]:
x_bit = 4 
captured = {}
def pre_hook(module, inputs):
    captured['x'] = inputs[0].detach().clone()
    
def forward_hook(module, inputs, output):
    captured['y'] = output.detach().clone()
    
input_handle = layer.register_forward_pre_hook(pre_hook)
output_handle = layer.register_forward_hook(forward_hook)

with torch.no_grad():
    model(input_batch.cuda())

input_handle.remove()
output_handle.remove()

x = captured['x']
psum_ref = captured['y'] 

x_alpha  =  layer.act_alpha.item()
x_delta = x_alpha / (2**x_bit - 1)

act_quant_fn = act_quantization(x_bit) 
x_q = act_quant_fn(x, x_alpha)        

x_int = x_q / x_delta
print(x_int[0])

tensor([[[ 0.0000,  0.0000,  4.0000,  4.0000],
         [ 0.0000,  3.0000, 10.0000,  7.0000],
         [ 0.0000,  7.0000, 13.0000,  7.0000],
         [ 0.0000,  2.0000,  3.0000,  1.0000]],

        [[ 0.0000,  0.0000,  1.0000,  0.0000],
         [ 0.0000,  2.0000,  1.0000,  0.0000],
         [ 3.0000,  7.0000,  8.0000,  0.0000],
         [ 1.0000,  0.0000,  0.0000,  0.0000]],

        [[ 2.0000,  0.0000,  0.0000,  9.0000],
         [ 1.0000,  0.0000,  2.0000, 12.0000],
         [ 0.0000,  0.0000,  0.0000,  5.0000],
         [ 4.0000,  8.0000,  7.0000,  5.0000]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  1.0000]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000,  0.0000],
   

In [158]:
#Verifying that manual convolution with modified convolution layer matches expected output
active_in = info["active_input"]
active_out = info["active_output"]

new_conv = nn.Conv2d(len(active_in), len(active_out), kernel_size=3, stride=1, padding=1, bias=False).cuda()

x_int_active = x_int[:, active_in, :, :]

with torch.no_grad():
    new_conv.weight.copy_(weight_int[active_out][:, active_in, :, :])
    psum_int = new_conv(x_int_active)

with torch.no_grad():
    psum_new = psum_int * x_delta * w_delta


psum_new_full = torch.zeros_like(psum_ref)
psum_new_full[:, active_out, :, :] = psum_new

with torch.no_grad():
    difference = (psum_new_full - psum_ref).abs()
    
print("Verifying I can reproduce fully pruned model with modified convolution layer with IC and OC = unpruned channels")
print("Mean absolute error compared to original output:", difference.mean().item())
print("Maximum absolute error:", difference.max().item())

Verifying I can reproduce fully pruned model with modified convolution layer with IC and OC = unpruned channels
Mean absolute error compared to original output: 1.2327002529843867e-08
Maximum absolute error: 7.152557373046875e-07


In [159]:

conv = model.features[27]

active_in = info["active_input"]
active_out = info["active_output"]

max_error = [0.0]
def compact_forward(module, inputs, output):
    # Quantize input activations
    x_q = module.act_alq(inputs[0], module.act_alpha)

    # Get masked quantized weights from current forward pass
    w_q = module.weight_q

    # Compact convolution using surviving channels only
    y_compact = F.conv2d(
        x_q[:, active_in, :, :],
        w_q[active_out][:, active_in, :, :],
        bias=None,
        stride=module.stride,
        padding=module.padding,
        dilation=module.dilation,
        groups=1
    )

    # Restore original 8-channel output shape
    y_full = torch.zeros_like(output)
    y_full[:, active_out, :, :] = y_compact

    # Compare compact output against original convolution
    error = (y_full - output).abs().max().item()
    max_error[0] = max(max_error[0], error)

    return y_full


# Original pruned model accuracy
original_acc = validate(testloader, model, criterion)

# Replace convolution output with compact result
handle = conv.register_forward_hook(compact_forward)

try:
    compact_acc = validate(testloader, model, criterion)
finally:
    handle.remove()

print("Original accuracy:", original_acc)
print("Compact accuracy:", compact_acc)
print("Maximum convolution error:", max_error[0])

assert max_error[0] < 1e-3

Test: [0/79]	Time 0.238 (0.238)	Loss 0.2479 (0.2479)	Prec 93.750% (93.750%)
 * Prec 91.070% 
Test: [0/79]	Time 0.045 (0.045)	Loss 0.2479 (0.2479)	Prec 93.750% (93.750%)
 * Prec 91.070% 
Original accuracy: 91.07
Compact accuracy: 91.07
Maximum convolution error: 0.0
